# Modelling

Logistic regression, converted into a cost-based three-action policy (approve / step-up / decline).

`model_train.parquet` / `model_test.parquet` are loaded from feature engineering, with cost assumptions from `config.py`.


## 1. Setup and load


In [1]:
cd ../


/Users/ann/Documents/Zempler/fraud_datasets


In [2]:
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, average_precision_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
import src.config as config


In [3]:
np.random.seed(config.RANDOM_SEED)

model_train = pd.read_parquet(config.TRAIN_PROCESSED_PATH)
model_test = pd.read_parquet(config.TEST_PROCESSED_PATH)

X_train, y_train = model_train[config.FEATURE_COLS], model_train[config.TARGET_COL]
X_test, y_test = model_test[config.FEATURE_COLS], model_test[config.TARGET_COL]

X_train.shape, X_test.shape


((1296675, 11), (555719, 11))

## 2. Baseline: let everything through

No model, no declines. Every fraud case still happens: the bank loses the full transaction amount, plus `FRAUD_CASE_HANDLING_COST` per case handled.


In [4]:
n_fraud_test = y_test.sum()
fraud_loss = model_test["amt"].to_numpy() + config.FRAUD_CASE_HANDLING_COST
is_fraud_test = y_test.to_numpy().astype(bool)

baseline_cost = fraud_loss[is_fraud_test].sum()

print(f"Fraud cases in test: {n_fraud_test}")
print(f"Baseline cost (let everything through): £{baseline_cost:,.2f}")


Fraud cases in test: 2145
Baseline cost (let everything through): £1,186,949.68


## 3. Temporal train / validation split

Temporal train/validation split (train on earlier transactions, validate on later ones).


In [5]:
TIME_COL = "trans_time"

time_values = pd.to_datetime(model_train[TIME_COL], errors="coerce")
if time_values.isna().all():
    raise ValueError(f"Could not parse timestamps from '{TIME_COL}'.")

order = np.argsort(time_values.to_numpy())
n_val = max(1, int(len(order) * 0.20))

train_idx, val_idx = model_train.index[order[:-n_val]], model_train.index[order[-n_val:]]

X_tr, y_tr = X_train.loc[train_idx], y_train.loc[train_idx]
X_val, y_val = X_train.loc[val_idx], y_train.loc[val_idx]

print(f"Train:      {X_tr.shape[0]:,} rows ({time_values.loc[train_idx].min()} → {time_values.loc[train_idx].max()})")
print(f"Validation: {X_val.shape[0]:,} rows ({time_values.loc[val_idx].min()} → {time_values.loc[val_idx].max()})")

Train:      1,037,340 rows (2019-01-01 00:00:18 → 2020-03-06 07:15:17)
Validation: 259,335 rows (2020-03-06 07:16:43 → 2020-06-21 12:13:37)


## 4. Feature check

`amt`/`age` are superseded by their banded versions.

Checking whether `time_since_last_trans_mins`, `trans_hour` and `merch_channel` are needed. Checked quickly with single passes (validation PR-AUC with vs without).

No class_weight="balanced" here, just plain logistic regression, so PR-AUC and predicted probabilities stay on the real class-frequency scale and are directly comparable across candidates.


In [6]:
CAT = set(config.CATEGORICAL_FEATURE_COLS)
NUM = set(config.NUMERIC_FEATURE_COLS)

def build_pipeline(features):
    cat = [c for c in features if c in CAT]
    num = [c for c in features if c in NUM]

    transformers = []
    if cat:
        transformers.append(("cat", OneHotEncoder(handle_unknown="ignore"), cat))
    if num:
        transformers.append(("num", Pipeline([
            ("impute", SimpleImputer(strategy="median")),
            ("scale", StandardScaler()),
        ]), num))

    return Pipeline([
        ("preprocess", ColumnTransformer(transformers)),
        ("clf", LogisticRegression(max_iter=1000, random_state=config.RANDOM_SEED)),
    ])

CANDIDATE_FEATURES = ["merch_channel", "trans_hour", "time_since_last_trans_mins"]
BASE = [f for f in config.FEATURE_COLS if f not in CANDIDATE_FEATURES]

def score(features):
    pipe = build_pipeline(features)
    pipe.fit(X_tr[features], y_tr)
    return average_precision_score(y_val, pipe.predict_proba(X_val[features])[:, 1])

selected = BASE
best_score = score(selected)
print(f"PR-AUC baseline: {best_score:.4f}\n")

for feature in CANDIDATE_FEATURES:
    with_feature = score(selected + [feature])
    print(f"PR-AUC without {feature}: {best_score:.4f}")
    print(f"PR-AUC with {feature}:    {with_feature:.4f}")

    if round(with_feature, 4) > round(best_score, 4):
        selected = selected + [feature]
        best_score = with_feature
        print(f"-> kept ({len(selected)} features so far)\n")
    else:
        print("-> dropped\n")

print(f"Selected features ({len(selected)}): {selected}")

PR-AUC baseline: 0.6256

PR-AUC without merch_channel: 0.6256
PR-AUC with merch_channel:    0.6256
-> dropped

PR-AUC without trans_hour: 0.6256
PR-AUC with trans_hour:    0.6320
-> kept (9 features so far)

PR-AUC without time_since_last_trans_mins: 0.6320
PR-AUC with time_since_last_trans_mins:    0.6317
-> dropped

Selected features (9): ['category', 'age_band', 'city_pop_band', 'amt_band', 'trans_dayofweek', 'is_evening_risk', 'is_late_night_risk', 'is_first_transaction', 'trans_hour']


## 5. Final model

Refit the selected-feature pipeline on the full training set.


In [7]:
model = build_pipeline(selected)
model.fit(X_train[selected], y_train);

## 6. Test-set model performance


ROC-AUC measures overall discrimination and PR-AUC focuses on precision and recall for the fraud class. PR-AUC is more meaningful because fraud highly imbalanced (~0.4%), making the model’s ability to identify genuine fraud among flagged transactions more relevant than overall ranking performance.

In [8]:
y_proba = model.predict_proba(X_test[selected])[:, 1]

print(f"ROC AUC:     {roc_auc_score(y_test, y_proba):.4f}")
print(f"PR AUC:      {average_precision_score(y_test, y_proba):.4f}  (vs {y_test.mean():.4f} for a random model)")

ROC AUC:     0.9619
PR AUC:      0.5771  (vs 0.0039 for a random model)


## 7. Coefficients


In [9]:
feature_names = model.named_steps["preprocess"].get_feature_names_out()
coefs = model.named_steps["clf"].coef_[0]

coef_table = (
    pd.DataFrame({"feature": feature_names, "coefficient": coefs})
    .assign(abs_coef=lambda d: d["coefficient"].abs()).sort_values("abs_coef", ascending=False).drop(columns="abs_coef")
)

coef_table.head(10)


,feature,coefficient
32,"cat__amt_band_(50.0, 100.0]",-8.131334
26,"cat__amt_band_(100.0, 200.0]",-5.970000
36,"cat__amt_band_(900.0, 1000.0]",4.081085
25,"cat__amt_band_(-0.001, 50.0]",-4.040702
27,"cat__amt_band_(1000.0, 1100.0]",3.977138
35,"cat__amt_band_(800.0, 900.0]",3.690413
34,"cat__amt_band_(700.0, 800.0]",3.228528
28,"cat__amt_band_(1100.0, 1200.0]",3.227557
2,cat__category_gas_transport,3.160589
4,cat__category_grocery_pos,2.779632


## 8. Three-action policy: approve / step-up / decline

The action is chosen purely from predicted fraud probability and transaction amount, by minimising expected cost across the three actions. Test labels only used afterwards, to score what actually happened.

Step-up abandonment cost is proxied by `WRONGFUL_DECLINE_COST` (the brief gives an abandonment *rate*, not a £ cost)


In [10]:
def expected_action_costs(probability, amounts, prior_declines):
    """Calculate expected costs for approve, step-up, and decline actions."""

    fraud_loss = amounts + config.FRAUD_CASE_HANDLING_COST

    attrition_penalty = np.where(prior_declines >= config.ATTRITION_DECLINE_THRESHOLD, config.ATTRITION_COST_PROXY, 0.0)

    decline_cost = (1 - probability) * (config.WRONGFUL_DECLINE_COST + attrition_penalty) + config.REVIEW_COST_PER_CASE

    cost_approve = probability * fraud_loss
    cost_stepup = (probability * (1 - config.STEP_UP_FRAUD_STOP_RATE) * fraud_loss + config.STEP_UP_COST
                   + (1 - probability) * config.STEP_UP_GENUINE_ABANDON_RATE * config.STEP_UP_ABANDON_COST_PROXY)

    return np.vstack([cost_approve, cost_stepup, decline_cost]).T


In [ ]:
def run_attrition_aware_policy(model_df, probability, y_true, customer_col="cc_num", date_col="trans_time", amount_col="amt"):
    """
    Apply a risk-based policy: aware of attrition risk to reduce repeatedly declining
    potential genuine transactions for the same cutomer within the same year
    """

    ordered = model_df.assign(fraud_probability=probability, is_fraud=y_true.to_numpy()).sort_values(date_col)
    timestamps = pd.to_datetime(ordered[date_col])

    action_names = ["approve", "stepup", "decline"]
    wrongful_decline_history = {}
    actions_out = np.empty(len(ordered), dtype=object)
    chosen_cost_out = np.empty(len(ordered))

    for i, (cust, ts, prob, amt, is_fraud) in enumerate(zip(
        ordered[customer_col], timestamps, ordered["fraud_probability"], ordered[amount_col], ordered["is_fraud"]
    )):
        hist = wrongful_decline_history.setdefault(cust, [])
        cutoff = ts - pd.Timedelta(days=config.ATTRITION_WINDOW_DAYS)
        hist[:] = [t for t in hist if t >= cutoff]

        costs = expected_action_costs(np.array([prob]), np.array([amt]), np.array([len(hist)]))[0]
        action_idx = costs.argmin()
        actions_out[i] = action_names[action_idx]
        chosen_cost_out[i] = costs[action_idx]

        if action_names[action_idx] == "decline" and not is_fraud:
            hist.append(ts)

    ordered = ordered.assign(action=actions_out, expected_cost=chosen_cost_out).sort_index()
    return ordered["action"].to_numpy(), ordered["expected_cost"].to_numpy()


In [12]:
actions, action_costs = run_attrition_aware_policy(model_test, y_proba, y_test, )

print("Test action counts:")
print(pd.Series(actions).value_counts())
print(f"\nTotal expected cost: £{action_costs.sum():,.2f}")
print(f"Let-everything-through baseline:                        £{baseline_cost:,.2f}")


Test action counts:
approve    536460
stepup      17513
decline      1746
Name: count, dtype: int64

Total expected cost: £124,240.75
Let-everything-through baseline:                        £1,186,949.68


# Findings Appendix

## 9. Out-of-sample results

Applying the policy once to the untouched test period. "Stopped" and "missed" fraud value for step-ups use the stated 90% challenge-effectiveness assumption; genuine impact is split into outright declines and step-ups (with an expected-abandonment estimate).


In [13]:
test_fraud = model_test[config.TARGET_COL].astype(bool)

declined_fraud = test_fraud & (actions == "decline")
stepup_fraud = test_fraud & (actions == "stepup")
declined_genuine = (~test_fraud) & (actions == "decline")
stepup_genuine = (~test_fraud) & (actions == "stepup")

# Fraud outcomes
fraud_cases_total = test_fraud.sum()
fraud_cases_stopped = declined_fraud.sum() + stepup_fraud.sum() * config.STEP_UP_FRAUD_STOP_RATE
fraud_case_stop_rate = fraud_cases_stopped / fraud_cases_total

fraud_value_total = model_test.loc[test_fraud, "amt"].sum()
fraud_value_stopped =  model_test.loc[declined_fraud, "amt"].sum() + model_test.loc[stepup_fraud, "amt"].sum() * config.STEP_UP_FRAUD_STOP_RATE
fraud_value_missed = fraud_value_total - fraud_value_stopped
fraud_value_stop_rate = fraud_value_stopped / fraud_value_total

# Genuine-customer impact
genuine_declined, genuine_stepup = declined_genuine.sum(), stepup_genuine.sum()
expected_genuine_abandonments = genuine_stepup * config.STEP_UP_GENUINE_ABANDON_RATE

operating_results = pd.Series({
    "fraud_value_stop_rate": fraud_value_stop_rate, "fraud_case_stop_rate": fraud_case_stop_rate,
    "fraud_value_stopped": fraud_value_stopped, "fraud_value_missed": fraud_value_missed,
    "genuine_transactions_declined": genuine_declined, "genuine_transactions_stepup": genuine_stepup,
    "expected_genuine_abandonments": expected_genuine_abandonments,
})

test_period_days = (
    pd.to_datetime(model_test["trans_time"]).max() - pd.to_datetime(model_test["trans_time"]).min()
).days
months_in_test = test_period_days / 30.44

rate_cols = ["fraud_value_stop_rate", "fraud_case_stop_rate"]
monthly_results = operating_results.copy()
monthly_results[[c for c in monthly_results.index if c not in rate_cols]] /= months_in_test
monthly_results = monthly_results.rename(lambda x: x if x in rate_cols else f"{x}_per_month")

print(f"Test period: {test_period_days} days (~{months_in_test:.1f} months)\n")
monthly_results


Test period: 193 days (~6.3 months)



fraud_value_stop_rate                           0.964638
fraud_case_stop_rate                            0.749091
fraud_value_stopped_per_month              172427.279690
fraud_value_missed_per_month                 6320.923726
genuine_transactions_declined_per_month       105.987979
genuine_transactions_stepup_per_month        2668.783627
expected_genuine_abandonments_per_month       213.502690
dtype: float64

Treating all declines as routed to manual review before being finalised.

In [14]:
review_cases_total = genuine_declined + declined_fraud.sum()  # all declines, genuine + fraud
review_cases_per_month = review_cases_total / months_in_test
review_cases_per_day = review_cases_per_month / 30.44

review_cost_total = review_cases_total * config.REVIEW_COST_PER_CASE
review_cost_per_month = review_cost_total / months_in_test

print(f"Declines routed to review: {review_cases_total:,} total, "
      f"~{review_cases_per_month:,.0f}/month, ~{review_cases_per_day:.1f}/day")
print(f"Review team capacity: {config.REVIEW_CAPACITY_PER_DAY}/day")
print(f"Utilisation: {review_cases_per_day / config.REVIEW_CAPACITY_PER_DAY:.1%} of capacity")
print(f"Review cost: £{review_cost_total:,.2f} total, £{review_cost_per_month:,.2f}/month")

Declines routed to review: 1,746 total, ~275/month, ~9.0/day
Review team capacity: 100/day
Utilisation: 9.0% of capacity
Review cost: £6,984.00 total, £1,101.52/month


## 10. Segment performance

Merchant category (primary business question, §12 Q3) and other candidate segments checked.


In [29]:
def segment_metrics(df, y_true, actions, segment_col):
    fraud = y_true.to_numpy().astype(bool)
    amounts = df["amt"].to_numpy()

    rows = []
    for segment, pos in df.groupby(segment_col, dropna=False, observed=True).indices.items():
        pos = np.asarray(pos)
        seg_fraud, seg_actions, seg_amounts = fraud[pos], actions[pos], amounts[pos]

        fraud_count, fraud_value = seg_fraud.sum(), seg_amounts[seg_fraud].sum()
        stopped_value = (
            seg_amounts[seg_fraud & (seg_actions == "decline")].sum()
            + seg_amounts[seg_fraud & (seg_actions == "stepup")].sum() * config.STEP_UP_FRAUD_STOP_RATE
        )

        genuine = ~seg_fraud
        affected = (
            (genuine & (seg_actions == "decline")).sum()
            + (genuine & (seg_actions == "stepup")).sum() * config.STEP_UP_GENUINE_ABANDON_RATE
        )

        rows.append({
            "segment": segment, "transactions": len(pos),
            "fraud_rate": fraud_count / len(pos), "fraud_count": fraud_count,       
            "fraud_value_stop_rate": stopped_value / fraud_value if fraud_value else np.nan,
            "genuine_affected_expected": affected,
        })

    return pd.DataFrame(rows).sort_values("transactions", ascending=False)


category_results = segment_metrics(model_test, y_test, actions, "category")
category_results.sort_values(["fraud_value_stop_rate", "genuine_affected_expected"], ascending=[True, False])

,segment,transactions,fraud_rate,fraud_count,fraud_value_stop_rate,genuine_affected_expected
13,travel,17449,0.002292,40,0.000000,119.20
7,kids_pets,48692,0.001335,65,0.000000,54.84
1,food_dining,39268,0.001375,54,0.000000,31.20
5,health_fitness,36674,0.001418,52,0.000000,22.00
3,grocery_net,19426,0.002111,41,0.000000,11.76
10,personal_care,39327,0.001780,70,0.072513,45.08
2,gas_transport,56370,0.002732,154,0.460663,281.12
6,home,52345,0.001280,67,0.738166,33.84
9,misc_pos,34574,0.002082,72,0.888264,173.76
0,entertainment,40104,0.001471,59,0.918154,50.28


In [16]:
age_results = segment_metrics(model_test, y_test, actions, "age_band")
age_results.sort_values(["fraud_value_stop_rate", "genuine_affected_expected"], ascending=[True, False])

,segment,transactions,fraud_rate,fraud_count,fraud_value_stop_rate,genuine_affected_expected
1,"(25, 35]",122790,0.003339,410,0.940705,440.48
2,"(35, 45]",116926,0.003113,364,0.944813,396.52
3,"(45, 55]",109372,0.003877,424,0.962086,417.48
0,"(0, 25]",54029,0.004072,220,0.970878,192.60
5,"(65, 100]",82987,0.004651,386,0.983287,326.24
4,"(55, 65]",69615,0.004898,341,0.985418,252.36


In [17]:
amt_results = segment_metrics(model_test, y_test, actions, "amt_band")
amt_results.sort_values(["fraud_value_stop_rate", "genuine_affected_expected"], ascending=[True, False]).head(5)

,segment,transactions,fraud_rate,fraud_count,fraud_value_stop_rate,genuine_affected_expected
1,"(100.0, 200.0]",74636,0.000724,54,0.000000,12.16
7,"(50.0, 100.0]",166804,0.000108,18,0.000000,0.00
0,"(-0.001, 50.0]",288930,0.001599,462,0.136770,319.52
6,"(400.0, 600.0]",4505,0.011987,54,0.909165,169.08
5,"(200.0, 400.0]",16678,0.033097,552,0.916814,658.40


In [18]:
city_results = segment_metrics(model_test, y_test, actions, "city_pop_band")
city_results.sort_values(["fraud_value_stop_rate", "genuine_affected_expected"], ascending=[True, False]).head(5)

,segment,transactions,fraud_rate,fraud_count,fraud_value_stop_rate,genuine_affected_expected
3,"(4680.0, 42384.0]",110451,0.004473,494,0.959822,458.88
0,"(1631.0, 4680.0]",110845,0.004790,531,0.962469,385.92
4,"(566.0, 1631.0]",112490,0.003991,449,0.965791,345.32
2,"(42384.0, 2906700.0]",110444,0.003332,368,0.968082,431.92
1,"(22.999, 566.0]",111489,0.002718,303,0.969978,403.64


In [19]:
first_txn_results = segment_metrics(model_test, y_test, actions, "is_first_transaction")
first_txn_results.sort_values(["fraud_value_stop_rate", "genuine_affected_expected"], ascending=[True, False])

,segment,transactions,fraud_rate,fraud_count,fraud_value_stop_rate,genuine_affected_expected
0,0,555703,0.003831,2129,0.964598,2025.68
1,1,16,1.000000,16,0.977715,0.00


****Worst-performing segments: amount band (primary), merchant category (secondary).****

By amount, the three worst are `(£50, £100]`, `(£100, £200]` and `(£0, £50]`, with 0.0%, 0.0% and 13.7% fraud-value stopped. They cover 530k of 555.7k test transactions (95%) and ~534 of 2,145 fraud cases (~25%), but most missed fraud is low-value.

****Why:**** These bands have below-baseline fraud rates (0.01–0.16% vs. 0.39% overall), so the model is correctly ranking them as low-risk. The gap is therefore policy economics, not model discrimination: at these amounts, intervention costs (fees, abandonment, attrition) can exceed expected fraud loss.

****Merchant category**** (§10) is a weaker secondary finding: `travel`, `kids_pets`, `food_dining`, `grocery_net`, and `health_fitness` all have 0.0% fraud-value-stop rates, with 40–65 confirmed fraud cases each. The same low-risk mechanism may apply(see §13 and Q3).

****Also checked, no material gap found:**** `age_band`, `city_pop_band`, `is_first_transaction`.


## 11. Customer risk

Which customers are showing up as highest-risk in the test period, ranked by their average predicted fraud probability.


In [20]:
customer_risk = (
    model_test.assign(fraud_probability=y_proba)
    .groupby("cc_num", observed=True)
    .agg(
        transactions=("fraud_probability", "size"), avg_fraud_probability=("fraud_probability", "mean"),
        max_fraud_probability=("fraud_probability", "max"), actual_fraud_cases=(config.TARGET_COL, "sum"),
    )
    .sort_values("avg_fraud_probability", ascending=False)
)
top_risk_customers = customer_risk.head(10)
top_risk_customers


,transactions,avg_fraud_probability,max_fraud_probability,actual_fraud_cases
cc_num,,,,
4883407061576,9,0.731632,0.935244,9
3588001568691267,14,0.706749,0.927675,14
586100864972,12,0.693639,0.944216,12
4417677808209716,11,0.688156,0.917608,11
372965408103277,10,0.685582,0.957726,10
2242176657877538,10,0.684332,0.933582,10
4295296907373,6,0.626929,0.942435,6
4352307151555405069,7,0.606774,0.919286,7
3550412175018089,11,0.599515,0.927328,11


In [21]:
def max_declines_in_window(times, window_days):
    times = sorted(times)
    max_count = 0
    start = 0
    for end in range(len(times)):
        while times[end] - times[start] > pd.Timedelta(days=window_days):
            start += 1
        max_count = max(max_count, end - start + 1)
    return max_count


declines = model_test.loc[(actions == "decline") & (~y_test.astype(bool))].copy()
declines["trans_time"] = pd.to_datetime(declines["trans_time"])

max_wrongful_declines_in_window = (
    declines.groupby("cc_num")["trans_time"]
    .apply(lambda s: max_declines_in_window(s, config.ATTRITION_WINDOW_DAYS))
)

n_wrongly_declined_gt2 = (max_wrongful_declines_in_window > config.ATTRITION_DECLINE_THRESHOLD).sum()

print(
    f"Customers wrongly declined more than {config.ATTRITION_DECLINE_THRESHOLD} times "
    f"within any {config.ATTRITION_WINDOW_DAYS}-day window: {n_wrongly_declined_gt2:,}"
)


Customers wrongly declined more than 2 times within any 365-day window: 3


## 12. Answers


### 1. What proportion of fraud can we stop?

In [22]:
print(f"   {fraud_value_stop_rate:.1%} of fraud value, {fraud_case_stop_rate:.1%} of fraud cases "
      f"(hard declines plus the assumed {config.STEP_UP_FRAUD_STOP_RATE:.0%} step-up stop rate).\n")

   96.5% of fraud value, 74.9% of fraud cases (hard declines plus the assumed 90% step-up stop rate).



### 2. How many genuine customers will we inconvenience to do it?

In [23]:
print(f"   {genuine_declined:,} genuine transactions declined outright, plus {genuine_stepup:,} sent to "
      f"step-up, of which ~{expected_genuine_abandonments:,.0f} are expected to abandon rather than "
      f"complete the challenge.\n")

   672 genuine transactions declined outright, plus 16,921 sent to step-up, of which ~1,354 are expected to abandon rather than complete the challenge.



### 3. Should we treat all merchant categories the same way?

In [33]:
spread = category_results["fraud_value_stop_rate"].max() - category_results["fraud_value_stop_rate"].min()
print(f"   Fraud-value stop rate spans a {spread:.0%} range across categories with enough volume to "
        f"judge. \n   Worth a bespoke policy only if that gap holds up out-of-time and the volume behind it is material.\n")

   Fraud-value stop rate spans a 99% range across categories with enough volume to judge. 
   Worth a bespoke policy only if that gap holds up out-of-time and the volume behind it is material.



### 4. Which of our customers are most likely to be defrauded?

In [35]:
print(f"   Top {len(top_risk_customers)} customers by average predicted fraud probability are listed "
          f"in section 11 (cc_num). \n")


   Top 10 customers by average predicted fraud probability are listed in section 11 (cc_num). 



### 13. Confidence, limitations and next improvements

**Confidence**

* **Model choice:** Medium. Logistic regression was chosen for interpretability, with three candidate features tested against a fixed base using the full temporal train/validation split. Temporal validation was also used. A non-linear model might improve PR-AUC, but it would need to be interpretable enough for real decline decisions.

* **£ savings:** Medium-low. Results depend on the cost assumptions and simulator, including fraud-handling and customer inconvenience costs.

* **Step-up economics:** Medium-low. Fraud-stop and abandonment rates come from the brief. Abandonment cost is estimated using wrongful-decline cost.

* **Category differences:** Medium/low unless there is enough data and a clear, stable difference. Note that this is a five-way tie at the bottom (§10).

* **Customer risk:** Low/medium. Rankings use average predicted probability and do not consider transaction value, recency, or low transaction counts. Results are also based on simulated data.

* **Attrition tracking:** Low/medium. The rolling wrongful-decline count (§11) assumes wrongful/genuine status is known in real time, which is optimistic (see decision log entry 6).

* **Category encoding:** Medium. `category` is kept at its full 14-level cardinality (see `01_feature_engineering.ipynb`, decision log entry 2). The reasoning is that L2 regularisation can handle the smaller groups. This has not been tested against a grouped alternative for this model.

**What could change this?**

Production fraud drift, observed step-up/3DS outcomes, a better estimate of abandonment cost, real customer/merchant behaviour data, and a realistic delayed model for when wrongful-decline status becomes known.

**Next improvements**

1. **Model monitoring and challenger testing:** track drift and compare models over time; relatively low effort.

2. **Real-time velocity features:** add rolling transaction counts and spend; high value but requires new infrastructure.



3. **Production feedback loop:** add chargebacks, disputes and step-up outcomes; high value but requires new data integrations.

### Decision log

1. **Temporal validation:** used because the model is intended for future production use.

2. **Logistic regression:** chosen for interpretability and time constraints; XGBoost was deferred.

3. **Lightweight feature check on the full temporal split:** three candidate features (`merch_channel`, `trans_hour`, `time_since_last_trans_mins`) were tested one at a time against a fixed base using the full `X_tr`/`X_val` split. There was no subsampling and no full search across every feature. Earlier drafts described this differently; this version matches the notebook.

4. **Three actions:** approve, step-up and decline. Low-risk transactions can be approved, high-risk transactions can be declined, and uncertain transactions can be stepped up for extra verification. This reduces unnecessary declines while still providing a way to handle transactions with some fraud risk.

5. **Cost assumptions:** abandonment cost was estimated using wrongful-decline cost because no separate estimate was provided. Review cost was included directly in the decline cost so it affects the action decision.

6. **Attrition tracking uses ground-truth labels at decision time, not a delayed proxy:** this is flagged as optimistic rather than fixed. A realistic delayed model would need an assumption about typical dispute/chargeback timing, which the brief does not provide.
